To train this model, click **Runtime** > **Run all**.

[![GitHub](https://img.shields.io/badge/GitHub-ART-blue?logo=github)](https://github.com/OpenPipe/ART)
[![Discord](https://img.shields.io/badge/Discord-Join-7289da?logo=discord&logoColor=white)](https://discord.gg/zbBHRUpwf4)
[![Docs](https://img.shields.io/badge/Docs-ART-green)](https://docs.art-e.dev/fundamentals/sft-training)

This notebook demonstrates how to fine-tune a model using **supervised fine-tuning (SFT)** with ART. We'll download a real dataset, train from a JSONL file, and optionally show how to do distillation from a larger teacher model.

Completions and metrics will be logged to [Weights & Biases](https://wandb.ai).

### Installation

In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !uv pip install openpipe-art[backend]==0.5.9 datasets --prerelease allow --no-cache-dir
else:
    try:
        import numpy

        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    try:
        import subprocess

        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except:
        is_t4 = False
    get_vllm, get_triton = (
        ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm", "triton")
    )
    !uv pip install --upgrade \
        openpipe-art[backend]==0.4.11 datasets pillow==11.3.0 protobuf==5.29.5 {get_vllm} {get_numpy} --prerelease allow --no-cache-dir
    !uv pip install -qqq {get_triton}

### Environment Variables

Set your `WANDB_API_KEY` to log metrics and checkpoints to [Weights & Biases](https://wandb.ai/home). This is optional — training will still work without it.

In [ ]:
import os

WANDB_API_KEY = ""
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

### Prepare Dataset

SFT training expects a JSONL file where each line has a `messages` array in the [OpenAI chat format](https://platform.openai.com/docs/api-reference/chat). The last message must be from the `assistant` role.

We'll use [HuggingFaceH4/no_robots](https://huggingface.co/datasets/HuggingFaceH4/no_robots), a high-quality dataset of 10k human-written instruction-following examples that already has a `messages` column in the right format.

In [ ]:
import json

from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/no_robots", split="train")

with open("train.jsonl", "w") as f:
    for row in dataset:
        f.write(json.dumps({"messages": row["messages"]}) + "\n")

print(f"Created train.jsonl with {len(dataset)} examples")

### Training

Create a model and train it from the JSONL file using `create_sft_dataset_iterator`. Each chunk logs its own metrics to W&B, so you can watch loss decrease over training.

In [ ]:
import json

from dotenv import load_dotenv

import art
from art.local import LocalBackend
from art.utils.sft import create_sft_dataset_iterator

load_dotenv()

backend = LocalBackend()
model = art.TrainableModel(
    name="sft-no-robots",
    project="sft-example",
    base_model="Qwen/Qwen3-30B-A3B-Instruct-2507",
)
await model.register(backend)

# Load trajectories from JSONL
trajectories = []
with open("train.jsonl") as f:
    for line in f:
        data = json.loads(line)
        trajectories.append(art.Trajectory(messages_and_choices=data["messages"]))

for chunk in create_sft_dataset_iterator(
    trajectories,
    epochs=3,
    batch_size=2,
    peak_lr=2e-4,
    schedule_type="cosine",
):
    await model.train_sft(chunk.trajectories, chunk.config)

print("Training complete!")

### Distillation (Alternative)

Instead of training from a static dataset, you can distill knowledge from a larger teacher model. The idea is simple: generate completions from the teacher, then train your model on those completions.

Uncomment the cell below to try it. You'll need an API key for the teacher model (this example uses [OpenRouter](https://openrouter.ai/)).

In [ ]:
# from openai import AsyncOpenAI
# from art.utils.sft import create_sft_dataset_iterator
#
# TEACHER_MODEL = "z-ai/glm-5"
#
# teacher_client = AsyncOpenAI(
#     api_key="your-openrouter-api-key",
#     base_url="https://openrouter.ai/api/v1",
# )
#
# prompts = [
#     "Explain recursion with a simple example.",
#     "What are the pros and cons of microservices?",
#     "How does a hash table work?",
# ]
#
# trajectories = []
# for prompt in prompts:
#     completion = await teacher_client.chat.completions.create(
#         model=TEACHER_MODEL,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     trajectories.append(art.Trajectory(
#         messages_and_choices=[
#             {"role": "user", "content": prompt},
#             {"role": "assistant", "content": completion.choices[0].message.content},
#         ],
#     ))
#
# for chunk in create_sft_dataset_iterator(trajectories, peak_lr=2e-4):
#     await model.train_sft(chunk.trajectories, chunk.config)

---

For more details, see the [SFT Training docs](https://docs.art-e.dev/fundamentals/sft-training). Questions? Join the [Discord](https://discord.gg/zbBHRUpwf4)!